## 7.3 – Dictionaries
### Hash Tables
The *associative array* is the abstract data type that you will recognise from its Python implementation: the dictionary. This is also called a *hash table* or *hash map*. Lots of names for the same thing: a structure which maps arbitrary *keys* to *values*.

In many units, this is where you would learn about the basic idea behind hash functions, how we can combine those with an array, how we run into collisions, and so on. But you've seen all that in the hash set!

Suppose we're given a key-value pair. The hash set already gives us the structure for how to turn an arbitrary key into an index for an array. If you have a fixed number of items and can design your hash to be perfect and unique, you do not need to store the key at all, you can just store the value in this location in the array. 

But the hash table is usually designed to support later additions, meaning collisions are possible, meaning we need to store the key also. So the way we'll do this is to store *two* items in each location in the underlying array, the key and the value. We can do this using a tuple, or we could write a class which contained fields `.key` and `.value`. I like the aesthetic of the latter, and if you are doing this in another language you might not have tuples, only `struct`, so let's do it that way.

In [1]:
class HashTableCell:
    def __init__(self, key, value):
        self.key = key
        self.value = value


class HashTable:
    def __init__(self, arr_size=100):
        self.arr = [None] * arr_size
        self.n = arr_size
        
    def add(self, key, value):
        key_index = hash(key) % self.n
        
        while self.arr[key_index] is not None and self.arr[key_index].key != key:
            key_index = (key_index + 1) % self.n
        
        self.arr[key_index] = HashTableCell(key, value)
        
    def get(self, key):
        key_index = hash(key) % self.n
        
        while self.arr[key_index] is not None and self.arr[key_index].key != key:
            key_index = (key_index + 1) % self.n
            
        if self.arr[key_index] is None:
            raise KeyError(f"No such key {key}") 
        
        return self.arr[key_index].value
    
    
my_table = HashTable()
my_table.add("ray", 5000)
my_table.add("ali", 3000)
my_table.add("sam", 2000)

print(my_table.get("sam"))

my_table.add("sam", 9000)

print(my_table.get("sam"))

2000
9000


Easy once you know the basics. Notice this implementation of the hash table naturally replaces existing values for known keys also, which is the standard behaviour.

#### Chaining
I also want to introduce a different collision resolution mechanic called *chaining*. 

The code above uses *linear probing* – we put the value into the next available space. 

With chaining, we just store a list of values at each space in the array! 

So we have a kind of 2D data structure, but each element could be a different length. Ideally we'll use a *linked list*, because we'll mostly be doing membership testing (search) which is going to be $O(n)$ whatever we do, so we want to avoid the cost of having to occasionally resize the array list.

***Exercise:*** Adapt the code above to use chaining. You can use a built in Python list, a `deque`, or reuse the LinkedList class from the previous notebook.



# Chaining vs Liner Probing

In the probing version, a collision sent you hunting for the next free slot in the array. In chaining, you never leave the slot. Each slot instead holds a whole collection of cells, and a collision just means that collection has more than one thing in it. So every operation becomes "hash to the right slot, then scan that one small collection." The outer while probing loop vanishes and an inner scan-the-bucket loop takes its place.

In [ ]:
from collections import deque


# ----------------------------------------------------------------------
# Shared cell type (key + value), same as the linear-probing version.
# ----------------------------------------------------------------------
class HashTableCell:
    def __init__(self, key, value):
        self.key = key
        self.value = value


# ======================================================================
# 1. Chaining with a built-in Python list
# ======================================================================
class HashTableList:
    def __init__(self, arr_size=100):
        # One empty list per slot. NOTE: a comprehension, NOT [[]] * n,
        # or every slot would share the SAME list object.
        self.arr = [[] for _ in range(arr_size)]
        self.n = arr_size

    def add(self, key, value):
        bucket = self.arr[hash(key) % self.n]
        for cell in bucket:
            if cell.key == key:          # key already here -> overwrite
                cell.value = value
                return
        bucket.append(HashTableCell(key, value))

    def get(self, key):
        bucket = self.arr[hash(key) % self.n]
        for cell in bucket:
            if cell.key == key:
                return cell.value
        raise KeyError(f"No such key {key}")


# ======================================================================
# 2. Chaining with a deque
# ======================================================================
class HashTableDeque:
    def __init__(self, arr_size=100):
        # new, empty deque — a double-ended queue with nothing in it yet
        self.arr = [deque() for _ in range(arr_size)]
        self.n = arr_size

    def add(self, key, value):
        bucket = self.arr[hash(key) % self.n]
        for cell in bucket:
            if cell.key == key:
                cell.value = value
                return
        bucket.append(HashTableCell(key, value))

    def get(self, key):
        bucket = self.arr[hash(key) % self.n]
        for cell in bucket:
            if cell.key == key:
                return cell.value
        raise KeyError(f"No such key {key}")


# ======================================================================
# 3. Chaining with our own singly linked list
# ======================================================================
class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None

    def find(self, key):
        node = self.head
        while node is not None:
            if node.key == key:
                return node             # match → leave the function entirely
            node = node.next            # no match → advance, only reached if the if was False
        return None                     # so can be None if the chain is empty to begin with or if you walked through the entire chain with While and on the last node next is none

    def add(self, key, value):
        node = self.find(key)
        if node is not None:             # overwrite existing
            node.value = value
        else:                            # prepend new node -> O(1)
            new_node = Node(key, value)
            new_node.next = self.head
            self.head = new_node


class HashTableLinkedList:
    def __init__(self, arr_size=100):
        self.arr = [LinkedList() for _ in range(arr_size)]
        self.n = arr_size

    def add(self, key, value):
        self.arr[hash(key) % self.n].add(key, value)

    def get(self, key):
        node = self.arr[hash(key) % self.n].find(key)
        if node is None:
            raise KeyError(f"No such key {key}")
        return node.value


# ----------------------------------------------------------------------
# Tests
# ----------------------------------------------------------------------
if __name__ == "__main__":
    tables = [
        ("list", HashTableList),
        ("deque", HashTableDeque),
        ("linked list", HashTableLinkedList),
    ]

    # Same sequence as the notebook's linear-probing example.
    for name, TableClass in tables:
        table = TableClass()
        table.add("ray", 5000)
        table.add("ali", 3000)
        table.add("sam", 2000)
        print(f"[{name}] sam =", table.get("sam"))      # 2000
        table.add("sam", 9000)                          # overwrite
        print(f"[{name}] sam =", table.get("sam"))      # 9000

    print("-" * 40)

    # Force collisions: a size-2 table makes several keys share a chain,
    # which is the whole point of chaining. All must still be retrievable.
    for name, TableClass in tables:
        table = TableClass(arr_size=2)
        pairs = {"ray": 5000, "ali": 3000, "sam": 2000, "kim": 7000, "joe": 1000}
        for k, v in pairs.items():
            table.add(k, v)
        ok = all(table.get(k) == v for k, v in pairs.items())
        print(f"[{name}] all 5 keys retrieved from a 2-slot table: {ok}")

### All 3 options share the same Big O complexity

| Case | What causes it | get | add |
|------|----------------|-----|-----|
| Best | Bucket empty or key is first cell | O(1) | O(1) |
| Average | Good hash, low load factor → short chains | O(1) | O(1) |
| Worst | Every key collides into one bucket → chain length n | O(n) | O(n) |

For all three that operation is the same: scanning the bucket to find the key. That scan is O(chain length) whether the bucket is a list, a deque, or a linked list — none of them lets you find a key without walking through the cells. The insertion step itself is O(1) for all three too, so nothing tips the complexity in any direction.

# Why for option 1 a list comprehension is needed?

The rule of thumb worth keeping: *-repetition is fine for immutable elements ([0] * n, ["x"] * n — you can never mutate an int or str in place, so sharing is invisible and harmless), but the instant the element is mutable — a list, a dict, a deque, your own class instance — reach for a comprehension instead. Same trap, incidentally, applies to [[]] * n inside 2D grids, which is where most people first get burned by it.

### What is a bucket?

so at one position you have. 1 bucket but it can hold multiple HashTableCells in the from of a list?

One position in self.arr = one bucket = one list. That list can hold zero, one, or many HashTableCell objects. Empty list means nothing has hashed there yet; one cell means a single key landed there; multiple cells means several keys collided to that index and are now chained together.


So there are really two layers stacked:


The outer layer is self.arr, a fixed-size array of n slots. The hash picks which slot (hash(key) % n).
The inner layer is the bucket at that slot — a list of cells. A linear scan picks which cell (for cell in bucket: if cell.key == key).


And each HashTableCell is just the key bundled with its value, which is why the cell has to store the key too: once several keys share a bucket, the value alone can't tell you whose it is — you need the key to identify the right one during the scan.


So your slot 0 from the last demo, [('ray', 5000), ('joe', 1000)], is literally one bucket (a list) holding two HashTableCell objects. That's chaining in a nutshell: the array gets you close, the list inside resolves the rest.

### Hash Table Complexity
The hash table has the same complexity considerations as the hash set. Provided it's big enough we get $O(1)$ average access efficiency. But if the load factor gets too high the performance degrades. As previously mentioned, most implementations will resize once they get too large, a feature we haven't included above.

What about *chaining* vs *probing*? You could write a dissertation on hash table conflict resolution, but in short, chaining has more consistent performance as load factor gets high, but it is slightly slower than probing if the load factor is kept within the normal range.

<img src="./resources/chain_probe.png" width=450 />

You can read a lot more about this subject online, such as [good old Wikipedia](https://en.wikipedia.org/wiki/Hash_table#Collision_resolution).

## What Next?
Once you're done with hash tables, head back to Engage to move onto the next section – a data structure that isn't already built in to Python!